<a href="https://colab.research.google.com/github/hamza26410/City-Library-After-School-Program-Project/blob/main/City%20Library%20After%20School%20Program.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Name: Hamza Ahmed Sayed Hassan AboZaid(حمزة أحمد سيد حسن أبو زيد)
# My ID Number: 31004260105918
# The Name Of The Project: 31004260105918_City Library After-School Program(31004260105918_City Library After-School Program.CSV)(مشروع تخرج شامل (Capstone Project))

# **Setup and Dataset preparation:**

In [ ]:
# 1.Load the dataset
import sqlite3
import pandas as pd
import json
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt

# **Task (1): Data Gathering and Combination: Part(1):**

In [ ]:
# 2.connect to the database file
conn = sqlite3.connect("level 3 final project library.db")
# 3.(read)get members table from the database
members_df = pd.read_sql_query("SELECT * FROM members", conn)
# 4.(get)join checkouts and members directly using SQL LEFT JOIN and read the query in the dataframe
sql_query = """
SELECT *
FROM checkouts
LEFT JOIN members ON checkouts.member_id = members.member_id
"""
stage1_df = pd.read_sql_query(sql_query, conn)
# 5. close database connection after getting the data
conn.close()

In [ ]:
# 6.(read)load the books json catalog file
books_df = pd.read_json("level 3 final project book catalog.json")

# 7.merge book details into our current data
stage2_df = pd.merge(stage1_df, books_df, on="book_id", how="left")

In [ ]:
# 8.read the html file for event signups
web_tables = pd.read_html("level 3 final project event sign up.html")
web_df = web_tables[0]

# 9.rename columns so they match our main data
web_df.columns = ["member_id", "book_id", "checkout_date"]

In [ ]:
# 10.create custom ids for web checkouts and set return date to empty
web_df["checkout_id"] = "WEB_" + (web_df.index + 1).astype(str)
web_df["return_date"] = None

In [ ]:
# 11.add member and book details to web signups
web_enriched = pd.merge(web_df, members_df, on="member_id", how="left")
web_enriched = pd.merge(web_enriched, books_df, on="book_id", how="left")

In [ ]:
# 12.combine all data sources into one final table
final_df = pd.concat([stage2_df, web_enriched], ignore_index=True)

# 13.save the combined dataset to csv file
final_df.to_csv("31004260105918_Library_task(1)_combined_data.csv", index=False)

# 14.check the result
final_df.head()

# **Task (1): Data Gathering and Combination: Part(2):**

In [ ]:
# 15.Connect to the database
conn = sqlite3.connect("level 3 final project library.db")

In [ ]:


# 16.Question (1): Get the total number of checkouts for each member (including members with zero checkouts)
# 17.Answer (1): Use LEFT JOIN to show all members even if they have 0 checkouts
sql_query_q1 = """
SELECT
    members.member_id,
    members.name,
    COUNT(checkouts.checkout_id) AS total_checkouts
FROM members
LEFT JOIN checkouts ON members.member_id = checkouts.member_id
GROUP BY members.member_id, members.name
"""

# 18.Run query and show result
q1_df = pd.read_sql_query(sql_query_q1, conn)
q1_df

In [ ]:
# 19.Question (2): Get book titles where the author's name starts with 'A' (showing the selected letter)
# 20.Answer (2): Use WHERE with LIKE 'A%' and add a constant column for the letter
sql_query_q2 = """
SELECT
    title,
    author,
    'A' AS selected_letter
FROM books
WHERE author LIKE 'A%'
"""
# 21.Run query and show result
q2_df = pd.read_sql_query(sql_query_q2, conn)
q2_df

In [ ]:
# 21.Question (3): Get the top 5 most checked-out books and their checkout count
# 22.Answer (3): JOIN books with checkouts, GROUP BY book, ORDER BY count DESC, LIMIT 5
sql_query_q3 = """
SELECT
    b.title,
    COUNT(c.checkout_id) AS checkout_count
FROM checkouts c
JOIN books b ON c.book_id = b.book_id
GROUP BY b.book_id, b.title
ORDER BY checkout_count DESC
LIMIT 5
"""

# 23.Run query and show result
q3_df = pd.read_sql_query(sql_query_q3, conn)
q3_df

In [ ]:
# 24.Question (4): Get top 10 members who checked out the most books
# 25.Answer (4): JOIN members with checkouts, GROUP BY member, ORDER BY total DESC, LIMIT 10
sql_query_q4 = """
SELECT
    m.name,
    COUNT(c.checkout_id) AS total_checkouts
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
GROUP BY m.member_id, m.name
ORDER BY total_checkouts DESC
LIMIT 10
"""

# 26.Run query and show result
q4_df = pd.read_sql_query(sql_query_q4, conn)
q4_df

In [ ]:
# 27.Question (5): Get checkouts for 'Downtown' neighborhood (skip first 10, get next 10)
# 28.Answer (5): JOIN checkouts with members, filter by neighborhood, ORDER BY date DESC, LIMIT 10 OFFSET 10
sql_query_q5 = """
SELECT
    c.*,
    m.neighborhood,
    'Downtown' AS selected_neighborhood
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE m.neighborhood = 'Downtown'
ORDER BY c.checkout_date DESC
LIMIT 10 OFFSET 10
"""

# 29.Run query and show result
q5_df = pd.read_sql_query(sql_query_q5, conn)
q5_df

In [ ]:
# 29.Close the database connection
conn.close()

In [ ]:
# 30.Prepare the full text content
text_content = """Task 1: SQL Queries and Detailed Answers

==================================================
Question 1:
Get the total number of checkouts for each member (including members with zero checkouts).

Answer 1:
We perform a LEFT JOIN between the 'members' table and the 'checkouts' table on 'member_id'. This ensures all members are listed, even if they have zero checkouts. We group by member_id and name, and count the checkouts.

SQL Query:
SELECT
    members.member_id,
    members.name,
    COUNT(checkouts.checkout_id) AS total_checkouts
FROM members
LEFT JOIN checkouts ON members.member_id = checkouts.member_id
GROUP BY members.member_id, members.name;

==================================================
Question 2:
Get book titles where the author's name starts with a specific letter (e.g., 'A') showing the letter.

Answer 2:
Selected Letter: 'A'
We filter the 'books' table using WHERE books.author LIKE 'A%'. We also add 'A' AS selected_letter to explicitly display the letter in the result.

SQL Query:
SELECT
    books.title,
    books.author,
    'A' AS selected_letter
FROM books
WHERE books.author LIKE 'A%';

==================================================
Question 3:
Get the top 5 most checked-out books and their checkout count.

Answer 3:
We JOIN 'checkouts' with 'books' on 'book_id', group by book title, count checkouts, order in descending order (DESC), and apply LIMIT 5.

SQL Query:
SELECT
    books.title,
    COUNT(checkouts.checkout_id) AS checkout_count
FROM checkouts
JOIN books ON checkouts.book_id = books.book_id
GROUP BY books.book_id, books.title
ORDER BY checkout_count DESC
LIMIT 5;

==================================================
Question 4:
Get top 10 members who checked out the most books.

Answer 4:
We JOIN 'checkouts' with 'members' on 'member_id', group by member name, count total checkouts, order from highest to lowest (DESC), and apply LIMIT 10.

SQL Query:
SELECT
    members.name,
    COUNT(checkouts.checkout_id) AS total_checkouts
FROM checkouts
JOIN members ON checkouts.member_id = members.member_id
GROUP BY members.member_id, members.name
ORDER BY total_checkouts DESC
LIMIT 10;

==================================================
Question 5:
Get checkouts for 'Downtown' neighborhood from newest to oldest (skip first 10, display next 10).

Answer 5:
Selected Neighborhood: 'Downtown'
Sorting: Newest to oldest (checkout_date DESC)
Pagination: Skip first 10 records (OFFSET 10), display next 10 records (LIMIT 10).
We JOIN 'checkouts' with 'members', filter by neighborhood = 'Downtown', order by checkout_date DESC, and apply LIMIT 10 OFFSET 10.

SQL Query:
SELECT
    checkouts.*,
    members.neighborhood,
    'Downtown' AS selected_neighborhood
FROM checkouts
JOIN members ON checkouts.member_id = members.member_id
WHERE members.neighborhood = 'Downtown'
ORDER BY checkouts.checkout_date DESC
LIMIT 10 OFFSET 10;
"""

# 31.Save the answers into the TXT file
new_filename = "31004260105918_Library_task(1)_sql_answers.txt"

with open(new_filename, "w") as file:
    file.write(text_content)

# 32.Download the file directly to your computer
from google.colab import files
files.download(new_filename)

# **Task (2): Data Integrity: Part(1):**

## **Data Exploration/Inspection:**



In [ ]:
# 33. load the combined dataset from task 1
file_path = "31004260105918_Library_task(1)_combined_data.csv"
df_task2 = pd.read_csv(file_path)

In [ ]:
# 34.Get column names / see the task I'm working on is right in front of me, and I know exactly what I’m working on before I start.
print("The Names of columns of the table of the DataFrame")
print(df_task2.columns)

In [ ]:
# 35. get a look at or an idea of ​​the DataFrame before starting work
# (First 15 rows)
print("Data Loaded From CSV file:")
print(df_task2.head(15))

In [ ]:
# 36. Get dimensions \ To show the DataFrame size (number of rows and columns) \ How many rows and columns?
print("Shape(Size)of the dataframe:")
print(df_task2.shape)

In [ ]:
# 37. Get DataFrame structure and Metadata / To find out the table structure of the DataFrame
print("Information of the dataframe:")
print(df_task2.info())

In [ ]:
# 38. Get STATISTICAL SUMMARY /Get the DataFrame's ===STATISTICAL SUMMARY=== (Digital statistics) to find errors before Fixing the DataFrame
print("Statistical Summary of the dataframe:")
print(df_task2.describe())


In [ ]:
# 39. Check if any column has any unique values in the table
print(df_task2.nunique() == len(df_task2))

# Explore The Unique Columns (member_id and isbn):

# 1. Get the number of unique values in each column
print("Unique values count per column:")
print(df_task2.nunique())

# 2. Show the list of unique values for a specific column (e.g., member_id)
print("Unique Member IDs:")
print(df_task2['member_id'].unique())

# 3. Get unique values and their frequencies
print("Value counts for member_id:")
print(df_task2['member_id'].value_counts())

In [ ]:
# 40. Get a semi complete overview of the DataFrame
print(df_task2)

# **Task (2): Data Integrity: Part(2):**

# **Data Cleaning((Fixing)(processing)(Scaning)):**


In [ ]:
# 41. Clean ISBN numbers

# 1. Remove dashes (-) from ISBN
df_task2['isbn'] = df_task2['isbn'].astype(str).str.replace('-', '')

# 2. Mark non-number ISBNs as Invalid_ISBN
df_task2.loc[~df_task2['isbn'].str.isdigit(), 'isbn'] = 'Invalid_ISBN'

print("ISBN Cleaning Done:")
print(df_task2['isbn'].value_counts())

In [ ]:
# 42. Handle missing values in member_id and other text columns

# Fill missing member_id with 'Unknown'
df_task2['member_id'] = df_task2['member_id'].fillna('Unknown')

print("Missing values after filling:")
print(df_task2.isna().sum())

In [ ]:
# 43. Check and drop duplicated rows

# 1. Count duplicates before deletion
print("Total duplicate rows:", df_task2.duplicated().sum())

# 2. Drop duplicates and keep the first occurrence
df_task2 = df_task2.drop_duplicates(keep='first')

print("Shape of DataFrame after removing duplicates:", df_task2.shape)

In [ ]:
# 44. Fix text formatting (Remove extra spaces and capitalize text)

# Remove spaces and fix capitalization for string columns
for col in df_task2.select_dtypes(include='object').columns:
    df_task2[col] = df_task2[col].astype(str).str.strip().str.title()

print("Text formatting fixed successfully!")

In [ ]:
# 45. Save the cleaned dataset to CSV and download it

# 1. Save cleaned data to a new CSV file
clean_filename = "31004260105918_Library_task(2)_cleaned_data.csv"
df_task2.to_csv(clean_filename, index=False)

# 2. Download to your computer directly
from google.colab import files
files.download(clean_filename)